# LEFT and RIGHT Outer Joins

## Overview
Outer joins preserve rows from one (or both) tables even when no match exists in the other. The unmatched side's columns are filled with `NULL`.

**JOIN types comparison:**

| Type | Returns |
|---|---|
| `INNER JOIN` | Only rows with a match in both tables |
| `LEFT JOIN` | All rows from the left table; NULLs for unmatched right columns |
| `RIGHT JOIN` | All rows from the right table; NULLs for unmatched left columns |
| `FULL OUTER JOIN` | All rows from both tables; NULLs on either side where no match |

**SQLite note:** SQLite supports LEFT JOIN and RIGHT JOIN (added in version 3.39.0, 2022). FULL OUTER JOIN is covered in the next notebook with a UNION-based workaround for older versions.

**The key insight:** A LEFT JOIN followed by `WHERE right_table.key IS NULL` finds rows in the left table with **no match** in the right — a very common pattern for finding gaps, orphans, and missing data.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id  INTEGER PRIMARY KEY,
    first_name  TEXT, last_name TEXT, dob TEXT, province TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT, province TEXT
);
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_provider_id INTEGER
);
CREATE TABLE encounters (
    enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER,
    dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER DEFAULT 0
);
CREATE TABLE diagnoses (
    diag_code TEXT PRIMARY KEY, description TEXT, category TEXT
);
CREATE TABLE referrals (
    referral_id INTEGER PRIMARY KEY, from_provider INTEGER, to_provider INTEGER,
    patient_id INTEGER, referral_date TEXT, reason TEXT
);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','NB',1),(2,'Liam','Chen','1972-11-04','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','ON',1),(4,'James','MacLeod','1955-01-30','NB',0),
  (5,'Sofia','Petrov','2001-09-15','BC',1),(6,'Noah','Williams','1968-06-08','AB',1),
  (7,'Mei','Zhang','1995-04-17','ON',1),(8,'Ethan','Tremblay','1980-12-01','QC',0),
  (9,'Priya','Nair','1977-08-25','BC',1),(10,'Marcus','Okafor','1993-05-19','ON',1),
  (11,'Dana','Leblanc','2000-02-14','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB'),(11,'Dr. James Wong','Oncology','BC'),
  (12,'Dr. Fatima Al-Rashid','Family Medicine','ON'),(13,'Dr. Ethan Tremblay','Orthopaedics','QC'),
  (14,'Dr. Priya Nair','Emergency Medicine','BC'),(15,'Dr. Marcus Okafor','Radiology','ON'),
  (16,'Dr. Unassigned','Neurology','NB');
INSERT INTO departments VALUES
  (1,'Cardiology','Building A',10),(2,'Oncology','Building B',11),
  (3,'Family Medicine','Building C',12),(4,'Orthopaedics','Building A',13),
  (5,'Emergency','Building D',14),(6,'Radiology','Building B',15),(7,'Neurology','Building C',16);
INSERT INTO diagnoses VALUES
  ('J18.9','Pneumonia, unspecified','Respiratory'),('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('Z00.0','General adult examination','Preventive'),('M17.1','Primary osteoarthritis, knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Cardiovascular'),
  ('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Neurological'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Musculoskeletal'),
  ('M54.5','Low back pain','Musculoskeletal');
INSERT INTO encounters VALUES
  (1,1,10,1,'2023-01-15','J18.9',1850.00,1),(2,2,10,1,'2023-02-20','I25.1',3200.00,1),
  (3,3,12,3,'2023-03-05','Z00.0',120.00,0),(4,4,13,4,'2023-03-18','M17.1',5500.00,1),
  (5,5,14,5,'2023-04-02','J06.9',95.00,0),(6,6,10,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,1,'2023-06-22','I10',2100.00,1),(8,8,12,3,'2023-07-14',NULL,80.00,0),
  (9,1,14,5,'2023-08-30','R55',410.00,0),(10,3,12,3,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,1,'2023-10-01','I48.0',1750.00,1),(12,10,13,4,'2023-11-03','S52.5',2200.00,1),
  (13,2,12,3,'2023-11-18',NULL,90.00,0),(14,5,13,4,'2023-12-05','M54.5',450.00,0);
INSERT INTO referrals VALUES
  (1,12,10,3,'2023-03-10','Chest pain follow-up'),(2,10,11,2,'2023-03-01','Suspected malignancy'),
  (3,14,10,9,'2023-09-05','AF workup'),(4,12,13,5,'2023-12-01','Back pain unresponsive to treatment'),
  (5,10,15,6,'2023-06-15','Imaging for chest pain');
""")
conn.commit()
print("Schema ready")
print(f"  patients: {conn.execute('SELECT COUNT(*) FROM patients').fetchone()[0]} rows (incl. Dana Leblanc, no encounters)")
print(f"  providers: {conn.execute('SELECT COUNT(*) FROM providers').fetchone()[0]} rows (incl. Dr. Unassigned, no encounters)")
print(f"  encounters: {conn.execute('SELECT COUNT(*) FROM encounters').fetchone()[0]} rows")


---
## LEFT JOIN: keep all rows from the left table

In [ ]:
# All patients, with their encounter count — including those with no encounters
print("=== All patients with encounter count (LEFT JOIN) ===")
q = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        p.province,
        COUNT(e.enc_id)                      AS encounters,
        ROUND(SUM(e.cost_cad), 2)            AS total_cost
FROM    patients     AS p
LEFT JOIN encounters AS e ON e.patient_id = p.patient_id
GROUP BY p.patient_id, p.first_name, p.last_name, p.province
ORDER BY encounters DESC, p.last_name
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("Dana Leblanc appears with 0 encounters — included because of LEFT JOIN.")
print("With INNER JOIN she would be excluded entirely.")


---
## Identifying rows with NO match: the anti-join pattern

In [ ]:
# Anti-join: patients who have NEVER had an encounter
print("=== Patients with no encounters (anti-join: LEFT JOIN + IS NULL) ===")
q = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        p.province,
        p.active
FROM    patients     AS p
LEFT JOIN encounters AS e ON e.patient_id = p.patient_id
WHERE   e.enc_id IS NULL
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Providers who have sent no referrals
print()
print("=== Providers who have sent no referrals ===")
q2 = """
SELECT  prov.provider_id,
        prov.full_name,
        prov.specialty
FROM    providers AS prov
LEFT JOIN referrals AS r ON r.from_provider = prov.provider_id
WHERE   r.referral_id IS NULL
ORDER BY prov.full_name
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# Diagnoses that appear in the lookup table but have never been used in an encounter
print()
print("=== ICD codes in diagnoses table never used in any encounter ===")
q3 = """
SELECT  d.diag_code, d.description, d.category
FROM    diagnoses    AS d
LEFT JOIN encounters AS e ON e.diag_code = d.diag_code
WHERE   e.enc_id IS NULL
ORDER BY d.category, d.diag_code
"""
print(pd.read_sql(q3, conn).to_string(index=False))


---
## Filtering on the joined table: WHERE vs ON

In [ ]:
# The placement of a filter — in ON vs WHERE — changes results for outer joins
print("=== LEFT JOIN with filter in ON clause (keeps all providers) ===")
q_on = """
SELECT  prov.full_name,
        prov.specialty,
        COUNT(e.enc_id)  AS cardiology_encounters
FROM    providers    AS prov
LEFT JOIN encounters AS e
    ON  e.provider_id = prov.provider_id
    AND e.dept_id = 1                     -- filter in ON: applied BEFORE the outer join
GROUP BY prov.provider_id, prov.full_name, prov.specialty
ORDER BY cardiology_encounters DESC
"""
print("Filter in ON (all providers kept, non-cardiology count is 0):")
print(pd.read_sql(q_on, conn).to_string(index=False))

print()
print("=== LEFT JOIN with filter in WHERE clause (only cardiology providers remain) ===")
q_where = """
SELECT  prov.full_name,
        prov.specialty,
        COUNT(e.enc_id)  AS cardiology_encounters
FROM    providers    AS prov
LEFT JOIN encounters AS e ON e.provider_id = prov.provider_id
WHERE   e.dept_id = 1                     -- filter in WHERE: applied AFTER join, eliminates NULLs
GROUP BY prov.provider_id, prov.full_name, prov.specialty
ORDER BY cardiology_encounters DESC
"""
print("Filter in WHERE (turns LEFT JOIN into an INNER JOIN effectively):")
print(pd.read_sql(q_where, conn).to_string(index=False))

print()
print("Rule: Conditions that filter the joined table belong in ON if you want to keep")
print("all left-table rows. Moving them to WHERE eliminates unmatched rows.")


---
## RIGHT JOIN and equivalence to LEFT JOIN

In [ ]:
# RIGHT JOIN: keep all rows from the RIGHT table
# Every RIGHT JOIN can be rewritten as a LEFT JOIN by swapping table order
print("=== RIGHT JOIN: all providers, with encounter counts ===")
q = """
SELECT  prov.full_name,
        prov.specialty,
        COUNT(e.enc_id)  AS encounters
FROM    encounters   AS e
RIGHT JOIN providers AS prov ON e.provider_id = prov.provider_id
GROUP BY prov.provider_id, prov.full_name, prov.specialty
ORDER BY encounters DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("=== Equivalent LEFT JOIN (table order swapped) ===")
q2 = """
SELECT  prov.full_name,
        prov.specialty,
        COUNT(e.enc_id)  AS encounters
FROM    providers    AS prov
LEFT JOIN encounters AS e ON e.provider_id = prov.provider_id
GROUP BY prov.provider_id, prov.full_name, prov.specialty
ORDER BY encounters DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))
print()
print("Results are identical. RIGHT JOIN is rarely needed — prefer LEFT JOIN for readability.")
print("SQLite added RIGHT JOIN support in v3.39.0 (2022); older versions only support LEFT JOIN.")

conn.close()


---
## Common Pitfalls

**1. Putting a filter on the joined table in WHERE instead of ON**
`LEFT JOIN t2 ON t1.id = t2.id WHERE t2.status = 'active'` eliminates all rows where `t2.status` is NULL — which includes all unmatched left rows. This silently converts your LEFT JOIN into an INNER JOIN. If you want to filter joined rows while keeping all left rows, put the condition in ON: `LEFT JOIN t2 ON t1.id = t2.id AND t2.status = 'active'`.

**2. Anti-join WHERE condition on wrong column**
`WHERE e.enc_id IS NULL` correctly finds patients with no encounters. But `WHERE e.patient_id IS NULL` would also work — however `WHERE e.cost_cad IS NULL` would not, because a matched encounter could legitimately have a NULL cost. Always use the primary key column of the joined table for the IS NULL check.

**3. COUNT(*) includes NULL rows from outer joins; COUNT(col) does not**
After a LEFT JOIN, `COUNT(*)` counts all rows including those with no match (NULLs in joined columns). `COUNT(e.enc_id)` counts only rows where `enc_id` is not NULL — i.e., only matched rows. Use `COUNT(joined_table.pk)` to get meaningful zero-counts for unmatched rows.

**4. SUM() of NULLs returns NULL, not zero**
`SUM(e.cost_cad)` returns NULL for a patient with no encounters, not 0. Use `COALESCE(SUM(e.cost_cad), 0)` if you want a numeric zero for unmatched rows.

**5. RIGHT JOIN is supported in SQLite only from v3.39.0 (July 2022)**
Older SQLite versions (still common in many environments) will raise a syntax error on RIGHT JOIN. Rewrite as LEFT JOIN with tables swapped — they are logically equivalent and LEFT JOIN is universally supported.

**6. Chaining multiple LEFT JOINs can produce unexpected row multiplication**
If table B has multiple matching rows for table A, and table C also has multiple matching rows for table A, a chain of LEFT JOINs will produce A×B×C rows per A row. Check cardinality carefully and consider aggregating intermediate results before joining.


---
*sql_methods_library - Samantha McGarrigle*